In [ ]:
import os
import time
from google.colab import drive

# Google Drive remount safely
drive.mount('/content/drive', force_remount=True)

# Direct Drive Dataset Path
DATASET_ROOT = "/content/drive/MyDrive/CSE720/EyeGAN"

if os.path.exists(DATASET_ROOT):
    print(f"Dataset directory found at: {DATASET_ROOT}")
else:
    print(f"WARNING: Directory {DATASET_ROOT} not found! Please check your Google Drive path.")

Mounted at /content/drive
Dataset directory found at: /content/drive/MyDrive/CSE720/EyeGAN


In [ ]:
import os
import time
from google.colab import drive

# Mount Google Drive safely
drive.mount('/content/drive', force_remount=True)

# Dataset path in Google Drive
DATASET_ROOT = "/content/drive/MyDrive/CSE720/EyeGAN"

if os.path.exists(DATASET_ROOT):
    print("Dataset directory found at:", DATASET_ROOT)
else:
    print("WARNING: Directory not found! Check your Drive directory structure.")

Mounted at /content/drive
Dataset directory found at: /content/drive/MyDrive/CSE720/EyeGAN


In [ ]:
import os
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import numpy as np

class Config:
    def __init__(self):
        self.dataset_path = DATASET_ROOT
        self.lr = 0.0002
        self.beta1 = 0.5
        self.beta2 = 0.999
        self.lambda_cls = 1.0
        self.lambda_gp = 10.0
        self.lambda_cycle = 10.0
        self.lambda_identity = 5.0
        self.lambda_perceptual = 1.0
        self.classifier_dir = "/content/drive/MyDrive/CSE720/output"
        self.benchmark_dir = os.path.join(self.classifier_dir, "benchmarks")

cfg = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class RobustDriveDataset(Dataset):
    def __init__(self, root_dir, img_size, split='train'):
        self.root_dir = root_dir
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

        self.samples = []
        if not os.path.exists(root_dir):
            raise FileNotFoundError(f"Path not found: {root_dir}")

        self.domains = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.domain_to_idx = {domain: i for i, domain in enumerate(self.domains)}

        for domain in self.domains:
            domain_path = os.path.join(root_dir, domain)
            for fname in os.listdir(domain_path):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(domain_path, fname), self.domain_to_idx[domain]))

        print(f"[{split}] Total images found: {len(self.samples)} across {len(self.domains)} domains.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        # Retry mechanism for Google Drive I/O drops
        for attempt in range(3):
            try:
                image = Image.open(img_path).convert('RGB')
                tensor_img = self.transform(image)
                return tensor_img, label, img_path
            except (OSError, IOError) as e:
                if attempt < 2:
                    time.sleep(0.5)
                else:
                    tensor_img = torch.zeros((3, self.img_size, self.img_size))
                    return tensor_img, label, img_path

def create_label_tensor(labels, batch_size, num_domains):
    # Device matching fix: generates random_shifts on the same GPU/CPU as labels
    random_shifts = torch.randint(1, num_domains, (batch_size,), device=labels.device)
    target_labels = (labels + random_shifts) % num_domains
    return target_labels

def create_same_label_tensor(labels, batch_size):
    return labels

In [ ]:
import torchvision.models as models

class Generator(nn.Module):
    def __init__(self, img_size=128, num_domains=5):
        super().__init__()
        self.label_emb = nn.Embedding(num_domains, img_size * img_size)
        self.img_size = img_size

        self.conv = nn.Sequential(
            nn.Conv2d(4, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x, c):
        c_emb = self.label_emb(c).view(-1, 1, self.img_size, self.img_size)
        x_in = torch.cat([x, c_emb], dim=1)
        return self.conv(x_in)


class Discriminator(nn.Module):
    def __init__(self, img_size=128, num_domains=5):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.out_src = nn.Conv2d(128, 1, kernel_size=3, stride=1, padding=1)
        self.out_cls = nn.Linear(128 * (img_size // 4) * (img_size // 4), num_domains)

    def forward(self, x):
        h = self.main(x)
        out_src = self.out_src(h)
        out_cls = self.out_cls(h.view(h.size(0), -1))
        return out_src, out_cls


class LossCalculator:
    def __init__(self, device, cfg=None):
        self.device = device
        self.cfg = cfg
        self.bce = nn.BCEWithLogitsLoss().to(device)
        self.ce = nn.CrossEntropyLoss().to(device)
        self.l1 = nn.L1Loss().to(device)

        # Updated VGG loading to clear deprecation warnings
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].to(device).eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg

    def adversarial_loss(self, logits, is_real):
        target = torch.ones_like(logits) if is_real else torch.zeros_like(logits)
        return self.bce(logits, target)

    def classification_loss(self, logits, labels):
        return self.ce(logits, labels)

    def reconstruction_loss(self, real, rec):
        return self.l1(real, rec)

    def identity_loss(self, real, ident):
        return self.l1(real, ident)

    def perceptual_loss(self, real, fake):
        return self.l1(self.vgg(real), self.vgg(fake))

    def gradient_penalty(self, D, real, fake):
        alpha = torch.rand(real.size(0), 1, 1, 1, device=self.device)
        interpolates = (alpha * real + ((1 - alpha) * fake)).requires_grad_(True)
        d_interpolates, _ = D(interpolates)
        fake_grad = torch.ones_like(d_interpolates, device=self.device)
        gradients = torch.autograd.grad(
            outputs=d_interpolates,
            inputs=interpolates,
            grad_outputs=fake_grad,
            create_graph=True,
            retain_graph=True,
            only_inputs=True,
        )[0]
        gradients = gradients.view(gradients.size(0), -1)
        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

In [ ]:
def benchmark_training(img_size, batch_size, n_epochs=2, num_domains=5):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    train_dataset = RobustDriveDataset(cfg.dataset_path, img_size, 'train')
    loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    G = Generator(img_size, num_domains=num_domains).to(device)
    D = Discriminator(img_size, num_domains=num_domains).to(device)
    g_opt = torch.optim.Adam(G.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
    d_opt = torch.optim.Adam(D.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))

    loss_calc = LossCalculator(device, cfg)

    epoch_times = []
    for epoch in range(n_epochs):
        t0 = time.time()
        for imgs, labels, _ in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            bs = imgs.size(0)
            target_labels = create_label_tensor(labels, bs, num_domains)
            same_labels = create_same_label_tensor(labels, bs)

            # Discriminator Step
            d_opt.zero_grad()
            real_out_src, real_out_cls = D(imgs)
            fake_imgs = G(imgs, target_labels)
            fake_out_src, _ = D(fake_imgs.detach())

            d_loss = (loss_calc.adversarial_loss(real_out_src, True) +
                      loss_calc.adversarial_loss(fake_out_src, False) +
                      cfg.lambda_cls * loss_calc.classification_loss(real_out_cls, labels) +
                      cfg.lambda_gp * loss_calc.gradient_penalty(D, imgs, fake_imgs.detach()))
            d_loss.backward()
            d_opt.step()

            # Generator Step
            g_opt.zero_grad()
            fake_out_src, fake_out_cls = D(fake_imgs)
            rec_imgs = G(fake_imgs, labels)
            identity_imgs = G(imgs, same_labels)

            g_loss = (loss_calc.adversarial_loss(fake_out_src, True) +
                      cfg.lambda_cls * loss_calc.classification_loss(fake_out_cls, target_labels) +
                      cfg.lambda_cycle * loss_calc.reconstruction_loss(imgs, rec_imgs) +
                      cfg.lambda_identity * loss_calc.identity_loss(imgs, identity_imgs) +
                      cfg.lambda_perceptual * loss_calc.perceptual_loss(imgs, fake_imgs))

            g_loss.backward()
            g_opt.step()

        epoch_times.append(time.time() - t0)

    peak_mem_mb = torch.cuda.max_memory_allocated() / (1024**2)
    n_params = sum(p.numel() for p in G.parameters()) + sum(p.numel() for p in D.parameters())

    del G, D, loss_calc
    torch.cuda.empty_cache()

    return {
        'img_size': img_size,
        'batch_size': batch_size,
        'mean_epoch_time_sec': float(np.mean(epoch_times)),
        'estimated_100_epoch_hours': float(np.mean(epoch_times)) * 100 / 3600,
        'peak_gpu_memory_MB': peak_mem_mb,
        'total_params': n_params,
    }

def benchmark_inference(img_size=128, num_domains=5, n_runs=50):
    G = Generator(img_size, num_domains=num_domains).to(device).eval()
    dummy_img = torch.randn(1, 3, img_size, img_size, device=device)
    dummy_label = torch.tensor([1], device=device)

    # Warmup
    for _ in range(10):
        _ = G(dummy_img, dummy_label)

    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.time()
            _ = G(dummy_img, dummy_label)
            torch.cuda.synchronize()
            latencies.append((time.time() - t0) * 1000)

    return {
        'img_size': img_size,
        'latency_ms': float(np.mean(latencies)),
        'throughput_fps': float(1000.0 / np.mean(latencies))
    }

In [ ]:
import pandas as pd

scalability_results = []
inference_results = []

# Probe resolutions
resolution_list = [64, 128]

for size in resolution_list:
    print(f"Profiling Resolution: {size}x{size}")
    res_train = benchmark_training(img_size=size, batch_size=8, n_epochs=2)
    scalability_results.append(res_train)

    res_infer = benchmark_inference(img_size=size)
    inference_results.append(res_infer)

train_df = pd.DataFrame(scalability_results)
infer_df = pd.DataFrame(inference_results)

print("\nTraining / Memory Scalability Results:")
print(train_df.round(3).to_string(index=False))

print("\nInference Latency / Throughput Results:")
print(infer_df.round(3).to_string(index=False))

# Safe save directory check
os.makedirs(cfg.benchmark_dir, exist_ok=True)

train_csv = os.path.join(cfg.benchmark_dir, 'training_scalability.csv')
infer_csv = os.path.join(cfg.benchmark_dir, 'inference_scalability.csv')

train_df.to_csv(train_csv, index=False)
infer_df.to_csv(infer_csv, index=False)

print(f"\nCSV Metrics saved successfully to: {cfg.benchmark_dir}")

Profiling Resolution: 64x64
[train] Total images found: 2500 across 5 domains.
Profiling Resolution: 128x128
[train] Total images found: 2500 across 5 domains.

Training / Memory Scalability Results:
 img_size  batch_size  mean_epoch_time_sec  estimated_100_epoch_hours  peak_gpu_memory_MB  total_params
       64           8              434.314                     12.064             193.333        590025
      128           8               98.186                      2.727             619.794       1142985

Inference Latency / Throughput Results:
 img_size  latency_ms  throughput_fps
       64       0.528        1895.302
      128       0.694        1441.520

CSV Metrics saved successfully to: /content/drive/MyDrive/CSE720/output/benchmarks
